In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import welch, butter, filtfilt

In [ ]:
def butter_highpass_filter(data, cutoff=1.0, fs=128, order=2):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    y = filtfilt(b, a, data)
    return y

In [ ]:
def extract_qeeg_metrics(df, fs):
    raw_signal = df.iloc[:, 0].values

    filtered_signal = butter_highpass_filter(raw_signal, cutoff=1.0, fs=fs)


    freqs, psd = welch(filtered_signal, fs=fs, nperseg=fs*4)

    bands = {
        'Delta': (1.0, 4),
        'Theta': (4, 8),
        'Alpha': (8, 13),
        'Beta': (13, 30)
    }

    metrics = {}
    total_idx = (freqs >= 1.0) & (freqs <= 30)
    total_power = np.trapezoid(psd[total_idx], freqs[total_idx])

    for band, (low, high) in bands.items():
        idx = (freqs >= low) & (freqs <= high)
        abs_pwr = np.trapezoid(psd[idx], freqs[idx])
        metrics[f'Abs_{band}'] = abs_pwr
        metrics[f'Rel_{band}'] = (abs_pwr / total_power) * 100

    metrics['Alpha_Beta_Ratio'] = metrics['Rel_Alpha'] / metrics['Rel_Beta']
    metrics['Theta_Alpha_Ratio'] = metrics['Rel_Theta'] / metrics['Rel_Alpha']
    metrics['Theta_Beta_Ratio'] = metrics['Rel_Theta'] / metrics['Rel_Beta']

    return metrics


In [ ]:
df_med = pd.read_csv("meditating.csv")
df_med = df_med[['FP2-F4']]


In [ ]:
df_think = pd.read_csv("thinking.csv")
df_think = df_think [["FP2-F4"]]

In [ ]:
df_med['FP2-F4'] = pd.to_numeric(df_med['FP2-F4'].astype(str).str.replace(',', '.'), errors='coerce')
df_think['FP2-F4'] = pd.to_numeric(df_think['FP2-F4'].astype(str).str.replace(',', '.'), errors='coerce')

df_med = df_med.dropna()
df_think = df_think.dropna()

meditation_results = extract_qeeg_metrics(df_med,fs=128)
baseline_results = extract_qeeg_metrics(df_think,fs=128)

comparison_df = pd.DataFrame([baseline_results, meditation_results],
                             index=['Baseline (Thinking)', 'Meditation']).T

print("\n==============================================")
print(f"      qEEG METRICS COMPARISON FOR FP2-F4           ")
print("==============================================")
print(comparison_df.round(4))
print("==============================================")

diff_alpha_beta = meditation_results['Alpha_Beta_Ratio'] / baseline_results['Alpha_Beta_Ratio']
diff_theta_alpha = meditation_results['Theta_Alpha_Ratio'] / baseline_results['Theta_Alpha_Ratio']
diff_theta_beta = meditation_results['Theta_Beta_Ratio'] / baseline_results['Theta_Beta_Ratio']

print(f"Alpha/Beta (Calmness Index):  {diff_alpha_beta:.2f}x higher during meditation.")
print(f"Theta/Alpha (Deep State Shift): {diff_theta_alpha:.2f}x higher during meditation.")
print(f"Theta/Beta (Focused Effort):   {diff_theta_beta:.2f}x higher during meditation.")


      qEEG METRICS COMPARISON FOR FP2-F4           
                   Baseline (Thinking)  Meditation
Abs_Delta                     751.6945   1270.5960
Rel_Delta                      13.0552      9.4306
Abs_Theta                    1431.0716   3532.8029
Rel_Theta                      24.8545     26.2213
Abs_Alpha                    2008.0959   4817.9758
Rel_Alpha                      34.8761     35.7601
Abs_Beta                     1566.9407   3851.6767
Rel_Beta                       27.2142     28.5880
Alpha_Beta_Ratio                1.2815      1.2509
Theta_Alpha_Ratio               0.7127      0.7333
Theta_Beta_Ratio                0.9133      0.9172
Alpha/Beta (Calmness Index):  0.98x higher during meditation.
Theta/Alpha (Deep State Shift): 1.03x higher during meditation.
Theta/Beta (Focused Effort):   1.00x higher during meditation.
